# Stage 2 — AI Enrichment (Claude)

Rakuten TV content discovery prototype — **Stage 2 only**: takes the raw TMDB metadata
produced by Stage 1 (`data/fetched_raw.json`, 21 items) and calls the Claude API to turn
it into clean, structured, consistent enriched metadata for each title.

Output: `data/enriched_final.json`, one record per item (content_id, title, year,
description, enriched_metadata), ready to be consumed by Stage 3 (search). This notebook
does not implement Stage 3.

## I. Import

### A. Libraries

In [13]:
import json
import os
import time
from pathlib import Path

import anthropic
from dotenv import load_dotenv

### B. Constants & config

In [14]:
load_dotenv()

ANTHROPIC_API_KEY = os.getenv("ANTHROPIC_API_KEY")
if not ANTHROPIC_API_KEY:
    raise RuntimeError(
        "ANTHROPIC_API_KEY is not set. Copy .env.example to .env and fill in your Anthropic API key."
    )

MODEL = "claude-sonnet-5"

INPUT_JSON_PATH = Path("data/fetched_raw.json")
OUTPUT_JSON_PATH = Path("data/enriched_final.json")

MAX_RETRIES = 1  # one retry on malformed/incomplete tool output, then fallback
REQUEST_TIMEOUT = 30  # seconds

client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)

## II. Data preparation

### A. Load stage 1 output

In [15]:
REQUIRED_KEYS = {"content_id", "title", "fetch_status"}


def load_fetched_data(json_path: Path) -> list:
    """Load stage 1 output and validate that required keys are present on every item."""
    with open(json_path, "r", encoding="utf-8") as f:
        items = json.load(f)

    for item in items:
        missing_keys = REQUIRED_KEYS - set(item.keys())
        if missing_keys:
            raise ValueError(f"Item {item.get('content_id', '?')} is missing required keys: {missing_keys}")

    return items


fetched_items = load_fetched_data(INPUT_JSON_PATH)
print(f"Loaded {len(fetched_items)} items from {INPUT_JSON_PATH}")

Loaded 21 items from data/fetched_raw.json


### B. Build the LLM input payload per item

In [16]:
def build_context_string(item: dict) -> str:
    """
    Assemble a clean textual summary of a stage 1 item, to be injected into the
    user message for enrichment. Handles missing/empty fields gracefully by
    falling back to "not available" instead of crashing.
    """
    title = item.get("title") or "Unknown title"
    year = item.get("year", "unknown year")
    media_type = item.get("media_type") or "unknown"

    overview = item.get("overview") or "No overview available."

    genres = item.get("genres") or []
    genres_str = ", ".join(genres) if genres else "not available"

    cast = item.get("cast") or []
    cast_str = ", ".join(cast) if cast else "not available"

    director = item.get("director") or "not available"

    keywords = item.get("keywords") or []
    keywords_str = ", ".join(keywords) if keywords else "not available"

    runtime = item.get("runtime")
    runtime_str = f"{runtime} minutes" if runtime else "not available"

    lines = [
        f"Title: {title} ({year})",
        f"Media type: {media_type}",
        f"Overview: {overview}",
        f"Genres: {genres_str}",
        f"Cast: {cast_str}",
        f"Director/Creator: {director}",
        f"Runtime: {runtime_str}",
        f"Keywords: {keywords_str}",
    ]

    return "\n".join(lines)

### C. Define the tool schema for structured output

In [17]:
ENRICHMENT_TOOL = {
    "name": "submit_enriched_metadata",
    "description": (
        "Submit structured enriched metadata for a single movie or TV show. "
        "target_audience must reflect scheduling-relevant audience segments useful for "
        "programming decisions (e.g. 'Family / Daytime', 'Late-night adult'), not generic "
        "marketing labels (avoid vague terms like 'everyone' or 'movie lovers')."
    ),
    "input_schema": {
        "type": "object",
        "properties": {
            "detailed_genres": {
                "type": "array",
                "items": {"type": "string"},
                "minItems": 1,
                "description": (
                    "Specific, nuanced genre labels beyond the broad TMDB genres "
                    "(e.g. 'Prison Drama', 'Psychological Thriller')."
                ),
            },
            "mood": {
                "type": "array",
                "items": {"type": "string"},
                "minItems": 1,
                "description": "The emotional tone(s) of the content (e.g. 'Inspiring', 'Tense', 'Melancholic').",
            },
            "themes": {
                "type": "array",
                "items": {"type": "string"},
                "minItems": 1,
                "description": "Core narrative themes explored in the content (e.g. 'friendship', 'injustice', 'redemption').",
            },
            "target_audience": {
                "type": "array",
                "items": {"type": "string"},
                "minItems": 1,
                "description": (
                    "Scheduling-relevant audience segments for programming decisions "
                    "(e.g. 'Family / Daytime', 'Late-night adult', 'Teens'), not generic "
                    "marketing labels."
                ),
            },
            "programming_slot_fit": {
                "type": "array",
                "items": {
                    "type": "string",
                    "enum": ["Prime time", "Late night", "Daytime", "Weekend afternoon", "Family slot"],
                },
                "minItems": 1,
                "description": "Which programming slot(s) this content fits best, from the fixed set of allowed slots.",
            },
            "similar_content_suggestions": {
                "type": "array",
                "items": {"type": "string"},
                "minItems": 1,
                "description": "Titles of similar movies or shows a viewer of this content might also enjoy.",
            },
            "content_warnings": {
                "type": "array",
                "items": {"type": "string"},
                "minItems": 0,
                "description": "Content warnings relevant to this title (e.g. 'Violence', 'Strong language'). Empty array if none apply.",
            },
            "viewing_context": {
                "type": "array",
                "items": {"type": "string"},
                "minItems": 1,
                "description": "Suggested viewing contexts (e.g. 'Evening watch', 'Family movie night', 'Background viewing').",
            },
        },
        "required": [
            "detailed_genres",
            "mood",
            "themes",
            "target_audience",
            "programming_slot_fit",
            "similar_content_suggestions",
            "content_warnings",
            "viewing_context",
        ],
    },
}

## III. Data processing

### A. System prompt

In [18]:
ENRICHMENT_SYSTEM_PROMPT = """You are a content metadata specialist working for the internal \
Content team of a streaming platform (catalog curation and channel programming, not a \
consumer-facing feature). For each title, you produce structured, consistent enriched \
metadata used for catalog search and programming decisions.

Rely primarily on the overview, genres, keywords, cast, and director provided to you. Do \
not invent specific plot details, characters, or events that are not supported by the \
provided input. If the input is sparse, keep your output more general rather than \
fabricating specifics.

You must always respond by calling the submit_enriched_metadata tool. Never respond in \
plain text."""

### B. Single-item enrichment function

In [19]:
def enrich_item(item: dict) -> dict:
    """
    Enrich a single stage 1 item via the Anthropic API, forcing the
    submit_enriched_metadata tool. Returns the tool's input dict, or None if
    the item was not successfully fetched in stage 1, or enrichment still
    fails after MAX_RETRIES retries.
    """
    if item.get("fetch_status") != "ok":
        return None

    context = build_context_string(item)
    user_message = f"Here is the content to enrich:\n\n{context}"

    last_error = None

    for attempt in range(MAX_RETRIES + 1):
        try:
            response = client.messages.create(
                model=MODEL,
                max_tokens=1024,
                system=ENRICHMENT_SYSTEM_PROMPT,
                tools=[ENRICHMENT_TOOL],
                tool_choice={"type": "tool", "name": "submit_enriched_metadata"},
                messages=[{"role": "user", "content": user_message}],
                timeout=REQUEST_TIMEOUT,
            )

            tool_use_block = next(
                (block for block in response.content if block.type == "tool_use"), None
            )
            if tool_use_block is None:
                raise ValueError("No tool_use block found in the response")

            return tool_use_block.input

        except Exception as e:
            last_error = e
            if attempt < MAX_RETRIES:
                time.sleep(0.5)

    print(f"[enrich_item] Enrichment failed for '{item.get('title')}' after {MAX_RETRIES + 1} attempt(s): {last_error}")
    return None


def process_all_items(items: list) -> list:
    """Sequentially enrich all items, printing progress. No parallelization needed for 21 items."""
    enriched_results = []

    for i, item in enumerate(items):
        print(f"[{i + 1}/{len(items)}] Enriching '{item.get('title')}'...")
        enriched_results.append(enrich_item(item))
        time.sleep(0.2)

    return enriched_results

### C. Assembly

In [20]:
def build_final_record(item: dict, enriched_metadata) -> dict:
    """Build the final stage 2 output record combining stage 1 raw data and enrichment."""
    return {
        "content_id": item.get("content_id"),
        "title": item.get("title"),
        "year": item.get("year"),
        "description": item.get("overview"),
        "enriched_metadata": enriched_metadata,
    }

## IV. Master / Export

### A. Run the full enrichment

In [21]:
enriched_metadata_list = process_all_items(fetched_items)

final_records = [
    build_final_record(item, enriched_metadata)
    for item, enriched_metadata in zip(fetched_items, enriched_metadata_list)
]

print(f"\nDone. Processed {len(final_records)} items.")

[1/21] Enriching 'The Shawshank Redemption'...
[enrich_item] Enrichment failed for 'The Shawshank Redemption' after 2 attempt(s): Error code: 401 - {'type': 'error', 'error': {'type': 'authentication_error', 'message': 'API key is invalid.'}, 'request_id': None}
[2/21] Enriching 'Inception'...
[enrich_item] Enrichment failed for 'Inception' after 2 attempt(s): Error code: 401 - {'type': 'error', 'error': {'type': 'authentication_error', 'message': 'API key is invalid.'}, 'request_id': None}
[3/21] Enriching 'Parasite'...
[enrich_item] Enrichment failed for 'Parasite' after 2 attempt(s): Error code: 401 - {'type': 'error', 'error': {'type': 'authentication_error', 'message': 'API key is invalid.'}, 'request_id': None}
[4/21] Enriching 'The Grand Budapest Hotel'...
[enrich_item] Enrichment failed for 'The Grand Budapest Hotel' after 2 attempt(s): Error code: 401 - {'type': 'error', 'error': {'type': 'authentication_error', 'message': 'API key is invalid.'}, 'request_id': None}
[5/21] Enr

### B. Validation summary

In [22]:
total = len(final_records)
succeeded = sum(1 for r in final_records if r["enriched_metadata"] is not None)
failed = total - succeeded

print(f"Total items:           {total}")
print(f"Successfully enriched: {succeeded}")
print(f"Failed / null:         {failed}")

if failed:
    print("\nFailed items:")
    for r in final_records:
        if r["enriched_metadata"] is None:
            print(f"  - {r['title']}")

Total items:           21
Successfully enriched: 0
Failed / null:         21

Failed items:
  - The Shawshank Redemption
  - Inception
  - Parasite
  - The Grand Budapest Hotel
  - Planet Earth II
  - Breaking Bad
  - Amélie
  - The Dark Knight
  - Moonlight
  - Spirited Away
  - Casablanca
  - Get Out
  - The Office
  - Mad Max: Fury Road
  - Jiro Dreams of Sushi
  - Chernobyl
  - 8½
  - Funny Games
  - Fleabag
  - Suspiria
  - Coherence


### C. Schema consistency check

In [23]:
EXPECTED_METADATA_KEYS = set(ENRICHMENT_TOOL["input_schema"]["required"])

print("Schema consistency check:")
for r in final_records:
    if r["enriched_metadata"] is None:
        continue

    actual_keys = set(r["enriched_metadata"].keys())
    if actual_keys == EXPECTED_METADATA_KEYS:
        print(f"  [PASS] {r['title']}")
    else:
        missing = EXPECTED_METADATA_KEYS - actual_keys
        extra = actual_keys - EXPECTED_METADATA_KEYS
        print(f"  [FAIL] {r['title']}: missing={missing or 'none'} extra={extra or 'none'}")

Schema consistency check:


### D. Export

In [24]:
OUTPUT_JSON_PATH.parent.mkdir(parents=True, exist_ok=True)

with open(OUTPUT_JSON_PATH, "w", encoding="utf-8") as f:
    json.dump(final_records, f, indent=2, ensure_ascii=False)

print(f"Saved {len(final_records)} items to {OUTPUT_JSON_PATH}")

Saved 21 items to data/enriched_final.json
